In [ ]:
%%bash
cd /kaggle/working && rm -rf SGG-Benchmark
git clone -q https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark

# The earlier runs cloned HEAD unpinned, which is the framework drift §7.4
# reports: arms trained weeks apart moved the human arm's pooled mR@100 by
# 0.033 with no label changed. Pin, and print the hash so this run is
# reproducible from the record rather than from a date.
COMMIT=$(git rev-list -1 --before="2026-08-21" HEAD)
git checkout -q "$COMMIT"
echo "PINNED COMMIT"
git log -1 --format="  %H%n  %ci%n  %s"

pip install -e . -q
pip install -q ultralytics hydra-core omegaconf
echo "INSTALL DONE"


In [ ]:
import pathlib
root = pathlib.Path("/kaggle/working/SGG-Benchmark/sgg_benchmark")
old = "from ultralytics.utils.plotting import feature_visualization"
marker = "feature_visualization = None  # removed in newer ultralytics"
NL = chr(10)
new = NL.join([
    "try:",
    "    from ultralytics.utils.plotting import feature_visualization",
    "except ImportError:",
    "    feature_visualization = None  # removed in newer ultralytics; only",
    "    # used by an optional debug path this run never enables",
])

patched = []
for f in root.rglob("*.py"):
    src = f.read_text(encoding="utf-8")
    if marker in src:
        continue
    if old in src:
        f.write_text(src.replace(old, new, 1), encoding="utf-8")
        patched.append(str(f.relative_to(root)))
print(f"patched {len(patched)} file(s):", patched)

import os, subprocess
os.chdir("/kaggle/working/SGG-Benchmark")
r = subprocess.run(["python", "-c",
    "from sgg_benchmark.modeling.detector import build_detection_model; print('IMPORT OK')"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1200:])
assert "IMPORT OK" in r.stdout, "import chain broken — paste the error above"

In [ ]:
import os, glob, json, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

yaml_hit = glob.glob("/kaggle/input/**/spatial_sgg_react.yaml", recursive=True)
assert yaml_hit, "base dataset not attached"
INPUT = os.path.dirname(yaml_hit[0])

auto = {}
for split in ("train", "val"):
    hits = [p for p in glob.glob(f"/kaggle/input/**/*{split}*annotations.auto.coco.json",
                                 recursive=True) if "auto-085" in p]
    assert hits, f"new {split} auto labels not found — is spatial-sgg-auto-085 attached?"
    auto[split] = hits[0]

print("base data :", INPUT)
for k, v in auto.items():
    print(f"new {k:5}:", v)

os.makedirs("datasets", exist_ok=True)
os.makedirs("configs/hydra/Spatial", exist_ok=True)
for d in ("spatial_sgg", "spatial_sgg_yolo"):
    if not os.path.isdir(f"datasets/{d}"):
        shutil.copytree(f"{INPUT}/{d}", f"datasets/{d}")
shutil.copy(f"{INPUT}/spatial_sgg_react.yaml", "configs/hydra/Spatial/react.yaml")

for split, src in auto.items():
    shutil.copy(src, f"datasets/spatial_sgg/{split}/_annotations.auto.coco.json")

for split, want in (("train", 121492), ("val", 18124)):
    d = json.load(open(f"datasets/spatial_sgg/{split}/_annotations.auto.coco.json"))
    n = len(d["rel_annotations"])
    bg = d["categories"][0]["name"] == "__background__"
    nr = d["rel_categories"][0]["name"] == "__no_relation__"
    print(f"{split}: {n} relations (expect {want}), bg0={bg}, norel0={nr}")
    assert n == want, f"{split}: got {n}, expected {want} — OLD LABELS, stop"
    assert bg and nr, f"{split}: background patch missing — training would give mR=0"

print("\nVERIFIED — new 0.85 labels staged, patch intact")

In [ ]:
import os, glob, shutil, yaml
os.chdir("/kaggle/working/SGG-Benchmark")
os.makedirs("checkpoints/BACKBONES", exist_ok=True)

found = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
if found:
    shutil.copy(found[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
    print("detector reused from", found[0])
else:
    print("no detector in input — training (~15 min)")
    yp = "datasets/spatial_sgg_yolo/data.yaml"
    d = yaml.safe_load(open(yp))
    d["path"] = os.path.abspath("datasets/spatial_sgg_yolo")
    yaml.safe_dump(d, open(yp, "w"))
    from ultralytics import YOLO
    YOLO("yolov8m.pt").train(data=yp, epochs=60, imgsz=640, batch=16,
                             project="det", name="yolov8m_spatial", verbose=False)
    src = max(glob.glob("runs/detect/det/yolov8m_spatial*/weights/best.pt"),
              key=os.path.getmtime)
    shutil.copy(src, "checkpoints/BACKBONES/yolov8m_spatial.pt")
    print("DETECTOR DONE ->", src)

In [ ]:
import subprocess, shutil, os, re, time, glob, json
os.chdir("/kaggle/working/SGG-Benchmark")
CKPT = "/kaggle/working/ckpt"
os.makedirs(CKPT, exist_ok=True)

# Ten seeds per arm. Three bounded the paired human-auto difference only to
# +/-0.070; ten takes that to roughly +/-0.020 and makes a non-inferiority
# statement possible instead of "undecided".
SEEDS = (42, 43, 44, 45, 46, 47, 48, 49, 50, 51)
ARMS = ("human", "auto")          # the pair the equivalence claim needs
BUDGET_H = 8.0                    # leave the rest of the session for evaluation

def stage(variant):
    """train/val take the arm's labels; test is ALWAYS human gold."""
    for split in ("train", "val"):
        shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.{variant}.coco.json",
                    f"datasets/spatial_sgg/{split}/_annotations.coco.json")
    shutil.copy("datasets/spatial_sgg/test/_annotations.human.coco.json",
                "datasets/spatial_sgg/test/_annotations.coco.json")

results, t_start, stopped = {}, time.time(), False

# Seed-major: each pass finishes a matched human/auto pair, so if the session
# dies the arms stay balanced and the surviving seeds are still comparable.
for seed in SEEDS:
    if stopped:
        break
    for variant in ARMS:
        tag = f"react_{variant}_s{seed}"
        if glob.glob(f"{CKPT}/{tag}/*.pth"):
            print(f"SKIP {tag} - already trained", flush=True)
            continue
        elapsed = (time.time() - t_start) / 3600
        if elapsed > BUDGET_H:
            print(f"\nSTOPPING at {elapsed:.1f}h to leave time for evaluation.")
            print("Re-run this cell in a fresh session to continue; it resumes.")
            stopped = True
            break

        stage(variant)
        os.system(f"rm -rf checkpoints/spatial/{tag}")
        cmd = ("python -u tools/relation_train_net_hydra.py "
               "--config-path ../configs/hydra/Spatial --config-name react "
               f"--task sgdet --save-best seed={seed} "
               f"output_dir=./checkpoints/spatial/{tag}")
        t0 = time.time()
        print("=" * 78, f"\nTRAIN {tag}  (elapsed {elapsed:.1f}h)\n", "=" * 78, flush=True)
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        out = r.stdout + "\n" + r.stderr
        print(out[-800:], flush=True)
        mrs = [float(x) for x in re.findall(r"Result for mR:\s*([\d.]+)", out)]
        results[tag] = max(mrs) if mrs else 0.0
        print(f"\n>>> {tag}: val mR={results[tag]:.4f} ({(time.time()-t0)/60:.1f} min)\n", flush=True)
        assert results[tag] > 0, f"{tag} produced mR=0 - stop and report"

        os.makedirs(f"{CKPT}/{tag}", exist_ok=True)
        for f in glob.glob(f"checkpoints/spatial/{tag}/*.pth") + \
                 glob.glob(f"checkpoints/spatial/{tag}/config.yml"):
            shutil.copy(f, f"{CKPT}/{tag}/")
        print(f"    banked -> {CKPT}/{tag}", flush=True)

shutil.copy("checkpoints/BACKBONES/yolov8m_spatial.pt", CKPT)
done = sorted(os.path.basename(p) for p in glob.glob(f"{CKPT}/react_*"))
print(f"\nTRAINED THIS SESSION: {results}")
print(f"CHECKPOINTS PRESENT ({len(done)}): {done}")


In [ ]:
import json, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")
for split in ["train", "val"]:
    shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.human.coco.json",
                f"datasets/spatial_sgg/{split}/_annotations.coco.json")
print("train/val staged to HUMAN (zero-shot reference; expect 94)")

TEST = "datasets/spatial_sgg/test"
full = json.load(open(f"{TEST}/_annotations.human.coco.json"))
shutil.copy(f"{TEST}/_annotations.human.coco.json", f"{TEST}/_annotations.full.coco.json")

def subset(group):
    keep = {im["id"] for im in full["images"] if im["file_name"].startswith(group + "_")}
    ann = [a for a in full["annotations"] if a["image_id"] in keep]
    akeep = {a["id"] for a in ann}
    rel = [r for r in full["rel_annotations"]
           if r["subject_id"] in akeep and r["object_id"] in akeep]
    d = dict(full); d["images"] = [im for im in full["images"] if im["id"] in keep]
    d["annotations"] = ann; d["rel_annotations"] = rel
    path = f"{TEST}/_annotations.{group}.coco.json"
    json.dump(d, open(path, "w"))
    print(f"  {group}: {len(d['images'])} images, {len(rel)} relations")
    return path

SLICES = {"full": f"{TEST}/_annotations.full.coco.json"}
for g in ["group_6", "group_7", "group_8"]:
    SLICES[g] = subset(g)

# convention-aligned: undo the inversion in groups 6 and 8
preds = [c["name"] for c in full["rel_categories"]]
FRONT, BEHIND = preds.index("in front of"), preds.index("behind")
img2grp = {im["id"]: im["file_name"].rsplit("_", 1)[0] for im in full["images"]}
ann2img = {a["id"]: a["image_id"] for a in full["annotations"]}
flipped, n = json.loads(json.dumps(full)), 0
for r in flipped["rel_annotations"]:
    if img2grp[ann2img[r["subject_id"]]] in {"group_6", "group_8"}:
        if r["predicate_id"] == FRONT: r["predicate_id"] = BEHIND; n += 1
        elif r["predicate_id"] == BEHIND: r["predicate_id"] = FRONT; n += 1
path = f"{TEST}/_annotations.aligned.coco.json"
json.dump(flipped, open(path, "w"))
assert n == 859, f"expected 859 flips, got {n}"
SLICES["full_aligned"] = path
print(f"flipped {n} relations -> full_aligned")
print("slices:", list(SLICES))

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, json, glob, shutil, gc, torch, logging, statistics as st
os.chdir("/kaggle/working/SGG-Benchmark")

SEEDS = (42, 43, 44, 45, 46, 47, 48, 49, 50, 51)
ARMS = ("human", "auto")

RUNS = {}
for variant in ARMS:
    for seed in SEEDS:
        tag = f"react_{variant}_s{seed}"
        ck = sorted(glob.glob(f"/kaggle/working/ckpt/{tag}/*.pth"))
        if not ck:
            print(f"no checkpoint for {tag} - skipping (training did not reach it)")
            continue
        RUNS[tag] = {"arm": variant, "seed": seed,
                     "cfg": f"/kaggle/working/ckpt/{tag}/config.yml", "ckpt": ck[-1]}

# "full" first for every run: it is the slice the equivalence bound needs, so a
# session that dies mid-evaluation still answers the main question.
ORDER = ["full", "full_aligned", "group_6", "group_7", "group_8"]
SL = {k: SLICES[k] for k in ORDER if k in SLICES}
print(f"{len(RUNS)} runs x {len(SL)} slices = {len(RUNS)*len(SL)} evaluations")

for _m in [m for m in list(sys.modules) if m.startswith("sgg_benchmark")]:
    del sys.modules[_m]

from omegaconf import OmegaConf
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer
from sgg_benchmark.data import make_data_loader
from sgg_benchmark.engine.inference import inference
try:
    from sgg_benchmark.utils.logger import setup_logger
    logger = setup_logger("sgg_benchmark", ".", 0, verbose="INFO", steps=True)
except Exception:
    logging.basicConfig(level=logging.INFO); logger = logging.getLogger("sgg_benchmark")

TEST = "datasets/spatial_sgg/test"
OUTJSON = "/kaggle/working/reeval_10seeds.json"
RESULTS = json.load(open(OUTJSON)) if os.path.exists(OUTJSON) else {}
print(f"resuming with {len(RESULTS)} result(s) already recorded")

def harvest(out, name, info, slice_name):
    f = f"{out}/eval_results_top_100.json"
    if not os.path.exists(f):
        return False
    d = json.load(open(f))
    zs = d.get("sgdet_zeroshot_recall", {}).get("100", [])
    RESULTS[f"{name}|{slice_name}"] = {
        "arm": info["arm"], "seed": info["seed"], "slice": slice_name,
        "R@100":  st.mean(d["sgdet_recall"]["100"]) if d.get("sgdet_recall") else None,
        "mR@100": d.get("sgdet_mean_recall", {}).get("100"),
        "F1@100": d.get("sgdet_f1_score", {}).get("100"),
        "zR@100": (st.mean(zs) if zs else 0.0),
        "n_zeroshot": len(zs),
    }
    json.dump(RESULTS, open(OUTJSON, "w"), indent=1)
    return True

for name, info in RUNS.items():
    todo = [s for s in SL if f"{name}|{s}" not in RESULTS]
    if not todo:
        print(f"SKIP {name} - all slices done"); continue

    cfg = OmegaConf.load(info["cfg"])
    model = build_detection_model(cfg).to(cfg.model.device)
    DetectronCheckpointer(cfg, model).load(info["ckpt"])
    model.eval()

    for slice_name in todo:
        out = f"./checkpoints/spatial/re_{name}_{slice_name}"
        if harvest(out, name, info, slice_name):
            print(f"SKIP {name}|{slice_name} - recovered from disk"); continue
        shutil.copy(SL[slice_name], f"{TEST}/_annotations.coco.json")
        os.makedirs(out, exist_ok=True)
        cfg.output_dir = out
        loader = make_data_loader(cfg, mode="test")[0]
        print("=" * 78, f"\nEVAL {name} on {slice_name} ({len(loader.dataset)} images)", flush=True)
        with torch.no_grad():
            inference(cfg, model, loader, dataset_name="SpatialRobot_test",
                      iou_types=("bbox", "relations"), box_only=False,
                      device=cfg.model.device, expected_results=[],
                      expected_results_sigma_tol=4, output_folder=out, logger=logger)
        harvest(out, name, info, slice_name)
        del loader
        gc.collect(); torch.cuda.empty_cache()

    del model
    gc.collect(); torch.cuda.empty_cache()
    print(f"FREED {name}; GPU {torch.cuda.memory_allocated()/1e9:.2f} GB\n", flush=True)

print("\n===== RESULTS =====")
print(f"{len(RESULTS)}/{len(RUNS)*len(SL)} evaluations complete -> {OUTJSON}")


In [ ]:
# The number this run exists to produce: the paired human-auto difference and
# how tightly ten seeds bound it. Three seeds gave [-0.070, +0.069].
import json, math, statistics as st

R = json.load(open("/kaggle/working/reeval_10seeds.json"))
pairs = []
for seed in range(42, 52):
    h = R.get(f"react_human_s{seed}|full")
    a = R.get(f"react_auto_s{seed}|full")
    if h and a:
        pairs.append((seed, h["mR@100"], a["mR@100"]))

print(f"matched seeds: {len(pairs)}")
print(f"{'seed':>5} {'human':>8} {'auto':>8} {'auto-human':>11}")
for s, h, a in pairs:
    print(f"{s:>5} {h:8.4f} {a:8.4f} {a-h:+11.4f}")

d = [a - h for _, h, a in pairs]
n = len(d)
if n >= 3:
    m, sd = st.mean(d), st.stdev(d)
    se = sd / math.sqrt(n)
    # two-sided 95% t critical value
    tcrit = {3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365,
             9: 2.306, 10: 2.262}.get(n, 1.96)
    lo, hi = m - tcrit * se, m + tcrit * se
    hm = [h for _, h, _ in pairs]
    am = [a for _, _, a in pairs]
    print(f"\nhuman mean {st.mean(hm):.4f}  spread {max(hm)-min(hm):.4f}")
    print(f"auto  mean {st.mean(am):.4f}  spread {max(am)-min(am):.4f}")
    print(f"\npaired mean difference {m:+.4f}   sd {sd:.4f}")
    print(f"95% CI  [{lo:+.4f}, {hi:+.4f}]   half-width {tcrit*se:.4f}")
    print(f"        = {tcrit*se/st.mean(hm)*100:.0f}% of the human arm's mean")
    if hi < 0.02 and lo > -0.02:
        print("\nNON-INFERIORITY SUPPORTED at a +/-0.02 margin.")
    else:
        print(f"\nNot yet inside +/-0.02; more seeds would be needed.")
